# Importing Libaries & Data Loading

In [ ]:
# ────────────────────────────────────────────────
# IMPORT LIBRARIES
# ────────────────────────────────────────────────
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report


In [ ]:
# ────────────────────────────────────────────────
# LOAD DATASET
# ────────────────────────────────────────────────

# Path to CSV file (replace with your local path if downloaded manually)
csv_file = 'dataset/student-mat.csv'  

# Load the CSV into a DataFrame
df = pd.read_csv(csv_file, sep=';')

# Preview the dataset
print("Dataset Shape:", df.shape)
display(df.head())

In [ ]:
print("Columns in DataFrame:", list(df.columns))


# Data Preprocessing

In [ ]:
# ────────────────────────────────────────────────
# DATA INSPECTION
# ────────────────────────────────────────────────

# Check for missing values
print("\nMissing Values:\n", df.isnull().sum())


In [ ]:
# Check data types
print("\nData Types:\n", df.dtypes)

In [ ]:
# Check unique values per column
print("\nUnique Values per Column:\n", df.nunique())

In [ ]:
print("CURRENT COLUMNS:", df.columns.tolist())

def assign_learning_path(g3):
    if g3 >= 15:
        return 'advanced'
    elif g3 >= 10:
        return 'standard'
    else:
        return 'remedial'

df['learning_path'] = df['G3'].apply(assign_learning_path)
df.drop(columns=['G3'], inplace=True)
print(df['learning_path'].value_counts())


# Exploratory Data Analysis (EDA)

In [ ]:
# ────────────────────────────────────────────────
# EDA: Class Distribution
# ────────────────────────────────────────────────
sns.countplot(x='learning_path', data=df)
plt.title('Distribution of Learning Paths')
plt.show()

In [ ]:
# ────────────────────────────────────────────────
# EDA: Numeric Feature Distributions
# ────────────────────────────────────────────────
numeric_cols = ['age', 'absences', 'G1', 'G2']

df[numeric_cols].hist(figsize=(10,8), bins=15)
plt.suptitle('Numeric Features Distribution')
plt.show()

In [ ]:
# ────────────────────────────────────────────────
# EDA: Correlation Heatmap (visualized)
# ────────────────────────────────────────────────
df_numeric = df.select_dtypes(include=np.number)
df_numeric_corr = df_numeric.corr()

plt.figure(figsize=(12,10))
sns.heatmap(df_numeric_corr, annot=True, cmap='coolwarm', fmt='.2f')
plt.title('Feature Correlation Matrix')
plt.show()


In [ ]:
# ────────────────────────────────────────────────
# EDA: Boxplots of Numeric Features by Learning Path (all side by side)
# ────────────────────────────────────────────────

numeric_cols = df.select_dtypes(include=np.number).columns.tolist()
n_cols = 3  # number of plots per row (adjust if needed)
n_rows = (len(numeric_cols) + n_cols - 1) // n_cols  # auto-calculate rows

fig, axes = plt.subplots(n_rows, n_cols, figsize=(n_cols * 5, n_rows * 5))

for idx, col in enumerate(numeric_cols):
    row = idx // n_cols
    col_idx = idx % n_cols
    sns.boxplot(x='learning_path', y=col, data=df, ax=axes[row, col_idx])
    axes[row, col_idx].set_title(f'{col} by Learning Path')

# Hide empty subplots (if any)
for j in range(idx + 1, n_rows * n_cols):
    fig.delaxes(axes.flatten()[j])

plt.tight_layout()
plt.show()


# Feature Engineering

In [ ]:
# Data Preparation for Modeling
# ────────────────────────────────────────────────
# ENCODING CATEGORICAL FEATURES
# ────────────────────────────────────────────────

# Identify categorical columns (exclude target)
categorical_cols = df.select_dtypes(include='object').columns.tolist()
categorical_cols.remove('learning_path')

print("Categorical columns to encode:", categorical_cols)

# Apply one-hot encoding
df_encoded = pd.get_dummies(df, columns=categorical_cols, drop_first=True)

print("Encoded DataFrame shape:", df_encoded.shape)

In [ ]:
# ────────────────────────────────────────────────
# ENCODE TARGET (learning_path)
# ────────────────────────────────────────────────
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
df_encoded['learning_path_encoded'] = le.fit_transform(df['learning_path'])

# Display mapping for clarity
print("Target class mapping:", dict(zip(le.classes_, le.transform(le.classes_))))

# ────────────────────────────────────────────────
# SPLIT FEATURES AND TARGET
# ────────────────────────────────────────────────
X = df_encoded.drop(columns=['learning_path', 'learning_path_encoded'])
y = df_encoded['learning_path_encoded']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

print("Train shape:", X_train.shape)
print("Test shape:", X_test.shape)

# Modeling

In [ ]:
# Modeling
# ────────────────────────────────────────────────
# DECISION TREE TRAINING
# ────────────────────────────────────────────────
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

# Initialize and train Decision Tree
clf = DecisionTreeClassifier(max_depth=4, random_state=42)  # you can adjust max_depth
clf.fit(X_train, y_train)

# Predictions
y_pred = clf.predict(X_test)

# Evaluation
print("\nAccuracy:", accuracy_score(y_test, y_pred))
print("\nClassification Report:\n", classification_report(y_test, y_pred, target_names=le.classes_))

In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay

# Visualized confusion matrix
cm = confusion_matrix(y_test, y_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=le.classes_)

plt.figure(figsize=(8,6))
disp.plot(cmap='Blues', values_format='d')
plt.title("Confusion Matrix")
plt.show()

In [ ]:
# Visualization
plt.figure(figsize=(20,10))
plot_tree(clf, feature_names=X.columns, class_names=le.classes_, filled=True, rounded=True)
plt.show()